# 03 — Cross-Validation: Single Split vs K-Fold, Stratification, and Leakage

**Overview:** show that a single train/test split is one noisy sample of performance;
contrast plain K-fold vs stratified K-fold on imbalanced data; demonstrate a leakage bug and
its fix inside a cross-validation loop.

**Required libraries:** `numpy`, `matplotlib`, `scikit-learn`.

Connects to: `01_classification_metrics.ipynb` and `11_generalization`'s train/validation
discipline. Closes out `12_ml_evaluation` — next is `13_feature_engineering`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **CV** | Cross-Validation | Repeatedly splitting data into train/validation folds to estimate generalisation. |

## 0. Environment

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                            weights=[0.9, 0.1], random_state=42)
print("class balance:", np.bincount(y) / len(y))


class balance: [0.896 0.104]


## 1. A single split gives one noisy sample of performance

⚠️ **Common Mistake:** reporting a single train/test split's score as THE performance number,
with no sense of how much it would vary under a different split.

In [2]:
single_split_scores = []
for seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    single_split_scores.append(model.score(X_test, y_test))

print("20 different single-split accuracies:", [round(s, 3) for s in single_split_scores])
print(f"mean={np.mean(single_split_scores):.3f}, std={np.std(single_split_scores):.3f}")
print("\nAny ONE of these 20 numbers, reported alone, would understate how much the estimate "
      "actually varies by chance of the split.")

assert np.std(single_split_scores) > 0   # there IS real variance across splits


20 different single-split accuracies: [0.96, 0.96, 0.96, 0.96, 0.953, 0.96, 0.953, 0.94, 0.967, 0.953, 0.96, 0.92, 0.96, 0.94, 0.94, 0.953, 0.947, 0.927, 0.947, 0.973]
mean=0.952, std=0.013

Any ONE of these 20 numbers, reported alone, would understate how much the estimate actually varies by chance of the split.


## 2. Cross-validation: averaging over many splits

💡 **Intuition:** `cross_val_score` is exactly the loop above, done systematically — each fold
is used as validation exactly once, so every data point contributes to exactly one score.

In [3]:
model = LogisticRegression(max_iter=1000)
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("5-fold CV scores:", cv_scores.round(3))
print(f"mean={cv_scores.mean():.3f}, std={cv_scores.std():.3f}")

assert len(cv_scores) == 5


5-fold CV scores: [0.94 0.93 0.96 0.95 0.95]
mean=0.946, std=0.010


## 3. Plain K-fold vs stratified K-fold under imbalance

⚠️ **Common Mistake:** using plain `KFold` on imbalanced data — some folds can end up with far
fewer (or zero) positive examples than others, making those folds' scores unreliable.

In [4]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Positive-class fraction per fold's VALIDATION set:\n")
print("Plain KFold:")
for fold_idx, (_, val_idx) in enumerate(kfold.split(X, y)):
    print(f"  fold {fold_idx}: {y[val_idx].mean():.3f}")

print("\nStratifiedKFold:")
stratified_fractions = []
for fold_idx, (_, val_idx) in enumerate(stratified_kfold.split(X, y)):
    frac = y[val_idx].mean()
    stratified_fractions.append(frac)
    print(f"  fold {fold_idx}: {frac:.3f}")

overall_rate = y.mean()
stratified_spread = np.std(stratified_fractions)
print(f"\noverall positive rate: {overall_rate:.3f}")
print(f"stratified folds' spread (std of per-fold rate): {stratified_spread:.4f} (should be small)")
assert stratified_spread < 0.03   # stratified folds should closely match the overall rate


Positive-class fraction per fold's VALIDATION set:

Plain KFold:
  fold 0: 0.110
  fold 1: 0.080
  fold 2: 0.070
  fold 3: 0.110
  fold 4: 0.150

StratifiedKFold:
  fold 0: 0.100
  fold 1: 0.100
  fold 2: 0.100
  fold 3: 0.110
  fold 4: 0.110

overall positive rate: 0.104
stratified folds' spread (std of per-fold rate): 0.0049 (should be small)


## 4. Leakage, demonstrated and fixed inside cross-validation

⚠️ **Common Mistake:** fitting a scaler on the FULL dataset before cross-validation, letting
every fold's "unseen" validation data influence the scaler via the other folds' overlap with
the full-data statistics.

In [5]:
from sklearn.model_selection import cross_val_score

# LEAKY: scaler fit on the entire dataset BEFORE cross-validation ever sees a fold boundary.
leaky_scaler = StandardScaler().fit(X)
X_leaky_scaled = leaky_scaler.transform(X)
leaky_cv_scores = cross_val_score(LogisticRegression(max_iter=1000), X_leaky_scaled, y, cv=5)

# CORRECT: wrap the scaler and model in a Pipeline, so the scaler is refit on ONLY each
# fold's training data, exactly as it would be fit at true training time.
from sklearn.pipeline import make_pipeline
correct_pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
correct_cv_scores = cross_val_score(correct_pipeline, X, y, cv=5)

print("leaky (scaler fit on full data):  mean =", round(leaky_cv_scores.mean(), 4))
print("correct (scaler fit per fold):    mean =", round(correct_cv_scores.mean(), 4))
print("\nOn this particular dataset the gap may be small -- the point isn't that leakage "
      "always inflates scores by a lot, it's that the LEAKY version is doing something subtly "
      "wrong regardless of how large the effect happens to be here. On a smaller dataset, or "
      "with a more data-dependent transform (e.g. feature selection), the gap would be larger.")

# Both should still be reasonable scores; the key lesson is the pipeline-based CORRECT pattern.
assert correct_cv_scores.mean() > 0.5


leaky (scaler fit on full data):  mean = 0.95
correct (scaler fit per fold):    mean = 0.95

On this particular dataset the gap may be small -- the point isn't that leakage always inflates scores by a lot, it's that the LEAKY version is doing something subtly wrong regardless of how large the effect happens to be here. On a smaller dataset, or with a more data-dependent transform (e.g. feature selection), the gap would be larger.


## 5. Connection to what's next

`13_feature_engineering` introduces exactly the kind of data-dependent transforms (encoding,
scaling, feature selection) that must be wrapped in a `Pipeline` like the one above to avoid
leakage during cross-validation or hyperparameter search (`14_hyperparameter_tuning`).

## 6. Interview questions
1. Why does a single train/test split understate the true uncertainty in a model's performance estimate?
2. Why is stratified K-fold specifically important under class imbalance?
3. What's wrong with fitting a `StandardScaler` on the entire dataset before running cross-validation?
4. Why does wrapping preprocessing and the model in a `Pipeline` fix the leakage problem during cross-validation?
5. If 5-fold CV scores have a high standard deviation, what does that tell you about the model or dataset?

## 7. Exercises
1. Repeat the single-split experiment with 100 different seeds and plot a histogram of the resulting accuracies.
2. Build a `Pipeline` with feature selection (e.g. `SelectKBest`) plus a classifier, and run it through `cross_val_score` to confirm it avoids the same leakage pattern.
3. Compare 5-fold vs 10-fold cross-validation's mean and standard deviation on the same dataset.

## 8. Summary

✅ A single split is one noisy sample; cross-validation (stratified, under imbalance) gives a
distribution you can actually trust; any data-dependent preprocessing must be refit per-fold,
not once on the full dataset, which a `Pipeline` enforces automatically. This closes
`12_ml_evaluation` — next: `13_feature_engineering`.